# 05 — Entraînement — le pipeline réel, ses artefacts et sa reproductibilité

**Projet** : Extraction d'entités nommées par spaCy — le tagger appris, corroboré par des règles

## Objectifs pédagogiques

1. Exécuter les pipelines du projet (`generate-data`, `train`) dans un bac à sable qui n'écrase rien.
1. Lire les artefacts d'un run : répertoire spaCy, métriques, fiche modèle, configuration résolue.
1. Recharger l'artefact et vérifier qu'il prédit **exactement** comme le modèle ajusté.
1. Prouver le déterminisme : deux ajustements à graine fixée donnent les mêmes mentions.

## 1. Mise en place

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.data.schemas import (  # noqa: E402
    ENTITY_LABELS,
)
from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un corpus réduit (320 messages) et un budget d'époques
# réduit : l'exécution reste rapide, et les valeurs absolues d'un notebook ne sont pas celles de la
# référence publiée par le README.
NB_DOCUMENTS = 320
NB_EPOCHS = 3

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                f"train.epochs={NB_EPOCHS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
                "train.callbacks.logging_every=100",
            ],
        )
    )

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
TARGET_COLUMN = str(MODEL_NODE.get("target", "label"))
ID_COLUMN = str(CONFIG.data.id_column or "msg_id")
SPLIT_COLUMN = str(CONFIG.data.group_column or "split")

# Les notebooks lisent **et écrivent** dans leur propre bac à sable : le corpus réduit et les
# artefacts des notebooks ne touchent ni `data/raw` ni `artifacts/`, donc ceux de `make train`
# restent ceux de la référence.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PROJECT_ROOT / "outputs" / "notebooks" / "data",
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.7)")
print(f"Algorithme servi  : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Types d'entités   : {len(ENTITY_LABELS)} — {', '.join(ENTITY_LABELS)}")
print(f"Bac à sable       : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticEntityCorpusGenerator
from src.data.loaders import EntityCorpusLoader

LOADER = EntityCorpusLoader(
    NB_PATHS,
    dataset_name=str(CONFIG.data.dataset_name),
    formats=tuple(CONFIG.data.formats),
    validation_enabled=bool(CONFIG.data.validation.raw),
    lazy_validation=bool(CONFIG.data.validation.lazy),
)

if LOADER.documents_path.exists():
    DOCUMENTS, SPANS = LOADER.load_corpus()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis le bac à sable : {len(DOCUMENTS)} messages, graine {METADATA['seed']}")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté **dans le bac à sable** pour que les notebooks suivants et les pipelines travaillent
    # sur exactement les mêmes fichiers.
    bundle = SyntheticEntityCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, SPANS, METADATA = bundle.documents, bundle.queries, dict(bundle.metadata)
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_annotations(SPANS)
    LOADER.save_metadata(METADATA)
    print(f"Bac à sable vide : corpus de {len(DOCUMENTS)} messages généré et persisté")

DOCUMENTS = DOCUMENTS.sort_values(ID_COLUMN).reset_index(drop=True)
SPANS = SPANS.sort_values([ID_COLUMN, "start", "end"]).reset_index(drop=True)

TRAIN_DOCUMENTS = LOADER.split("train", frame=DOCUMENTS)
VAL_DOCUMENTS = LOADER.split("val", frame=DOCUMENTS)
CALIBRATION_DOCUMENTS = LOADER.split("calibration", frame=DOCUMENTS)
TEST_DOCUMENTS = LOADER.split("test", frame=DOCUMENTS)
TRAIN_SPANS = LOADER.split_annotations("train", documents=DOCUMENTS, spans=SPANS)
VAL_SPANS = LOADER.split_annotations("val", documents=DOCUMENTS, spans=SPANS)
TEST_SPANS = LOADER.split_annotations("test", documents=DOCUMENTS, spans=SPANS)

print(
    "messages par split :",
    {
        name: len(frame)
        for name, frame in (
            ("train", TRAIN_DOCUMENTS),
            ("val", VAL_DOCUMENTS),
            ("calibration", CALIBRATION_DOCUMENTS),
            ("test", TEST_DOCUMENTS),
        )
    },
)
print(
    "mentions par split :",
    {
        name: len(frame)
        for name, frame in (("train", TRAIN_SPANS), ("val", VAL_SPANS), ("test", TEST_SPANS))
    },
)

## 2. Les pipelines du projet, dans un bac à sable

Le notebook appelle **les pipelines du projet**, pas une réimplémentation : ce qu'on lit ici est exactement ce que fait `python -m src.main mode=train`. Les artefacts partent dans `outputs/notebooks/`, donc ceux de `make train` ne sont jamais écrasés.

In [ ]:
from src.pipelines import DataGenerationPipeline, TrainPipeline

DATA_RESULT = DataGenerationPipeline(CONFIG, paths=NB_PATHS).run()
print(
    "generate-data :",
    "succès" if DATA_RESULT.succeeded else "échec",
    f"en {DATA_RESULT.duration_seconds:.2f}s",
)
for message in DATA_RESULT.messages:
    print("  -", message)

TRAIN_RESULT = TrainPipeline(CONFIG, paths=NB_PATHS).run()
print()
print(
    "train :",
    "succès" if TRAIN_RESULT.succeeded else "échec",
    f"en {TRAIN_RESULT.duration_seconds:.2f}s",
)
for message in TRAIN_RESULT.messages:
    print("  -", message)
assert TRAIN_RESULT.succeeded, TRAIN_RESULT.messages
OUTCOME = TRAIN_RESULT.payload

## 3. Les artefacts du run

L'artefact d'un pipeline spaCy est un **répertoire** : le pipeline complet (tokéniseur, vocabulaire, reconnaisseur d'entités, configuration), pas un fichier unique. C'est ce qui permet de le recharger sans reconstruire l'architecture à la main.

In [ ]:
ARTIFACT_DIR = NB_PATHS.models_dir / "tagger"
inside = sorted(
    path.relative_to(ARTIFACT_DIR) for path in ARTIFACT_DIR.rglob("*") if path.is_file()
)
print(f"répertoire du modèle : {ARTIFACT_DIR.relative_to(PROJECT_ROOT)}")
for name in inside:
    print("  -", name)
print()
for name in ("model_card.json", "resolved_config.json"):
    path = NB_PATHS.models_dir / name
    print(
        f"{name:24s} {'écrit' if path.exists() else 'ABSENT':6s} "
        f"({path.stat().st_size if path.exists() else 0} octets)"
    )
metrics_path = NB_PATHS.metrics_dir / "training_metrics.json"
print(
    f"{'training_metrics.json':24s} {'écrit' if metrics_path.exists() else 'ABSENT':6s} "
    f"({metrics_path.stat().st_size if metrics_path.exists() else 0} octets)"
)
assert ARTIFACT_DIR.is_dir() and metrics_path.exists()

**Ce qu'il faut retenir**

- Quatre artefacts répondent à quatre questions : le répertoire spaCy (comment prédire), les métriques (ce que ça vaut), la fiche modèle (comment c'est fait), la configuration résolue (avec quels réglages exactement).
- La configuration résolue est le seul moyen de rejouer un run : la configuration Hydra est faite de couches, et une couche qui change change tout.
- L'index de règles est **dans** l'artefact : à l'inférence, le modèle n'a pas besoin du corpus pour retrouver les surfaces qu'il a apprises.

## 4. Lire les métriques du run

Les métriques d'entraînement distinguent ce que le modèle a **vu** (`train_*`) de ce qu'il n'a pas vu (`val_*`). L'écart entre les deux est la première mesure de généralisation disponible.

In [ ]:
METRICS = pd.Series(OUTCOME.metrics).sort_values(ascending=False)
display(METRICS.to_frame("valeur").round(4))
print(
    f"messages d'entraînement : {OUTCOME.n_train_documents} "
    f"| entités vues : {OUTCOME.n_train_entities}"
)
print(
    f"messages de validation  : {OUTCOME.n_val_documents} "
    f"| entités de référence : {OUTCOME.n_val_entities}"
)
print(
    f"alignement des annotations : {OUTCOME.metrics['alignment_aligned_rate']:.4f} "
    f"({int(OUTCOME.metrics['alignment_ignored'])} mention(s) ignorée(s))"
)
display(OUTCOME.per_label.round(4))

In [ ]:
fit = OUTCOME.fit_result
print(f"ajustement : {fit.n_samples} messages, {fit.epochs} époque(s), {fit.duration_seconds:.2f}s")
print("paramètres effectifs :", fit.params)
print("extra du run        :", {key: fit.extra[key] for key in sorted(fit.extra)})

HISTORY = pd.DataFrame(OUTCOME.history)
display(HISTORY.round(4))
if "loss" in HISTORY.columns:
    HISTORY["loss"].plot(title="perte d'entraînement par époque", xlabel="époque")
    plt.tight_layout()

**Ce qu'il faut retenir**

- `alignment_ignored = 0` est un résultat du corpus, pas une chance : les annotations sont écrites avec le texte (notebook 03).
- La table par type est publiée **dès l'entraînement** : elle évite de découvrir au rapport final qu'un type a été sacrifié.
- Le budget d'époques du notebook est réduit : la perte ne descend donc pas autant que dans le run de référence.

## 5. Recharger et comparer

Un artefact qui ne se recharge pas ne se déploie pas. Le test compare les mentions **span par span**, pas une métrique moyenne : un écart d'un seul caractère doit apparaître.

In [ ]:
from src.models import build_model, load_model

RESTORED = load_model(ARTIFACT_DIR, config=CONFIG.model_dump())
print("artefact rechargé :", RESTORED.summary())
print("types servis      :", RESTORED.labels)

TEXTS = [str(text) for text in TEST_DOCUMENTS[TEXT_COLUMN]]
IDENTIFIERS = [str(value) for value in TEST_DOCUMENTS[ID_COLUMN]]

FIRST = build_model(CONFIG)
FIRST.fit(TRAIN_DOCUMENTS, TRAIN_SPANS)
COPY_DIR = FIRST.save(NB_PATHS.models_dir / "notebook_copy")
RELOADED = load_model(COPY_DIR, config=CONFIG.model_dump())

BEFORE = FIRST.predict(TEXTS, ids=IDENTIFIERS)
AFTER = RELOADED.predict(TEXTS, ids=IDENTIFIERS)
identical = [[mention.as_tuple() for mention in found] for found in BEFORE] == [
    [mention.as_tuple() for mention in found] for found in AFTER
]
print("mentions identiques après rechargement :", identical)
print("exemple :", [mention.as_tuple() for mention in BEFORE[0]])
assert identical, "un artefact rechargé doit prédire exactement comme le modèle ajusté"

In [ ]:
SECOND = build_model(CONFIG)
SECOND.fit(TRAIN_DOCUMENTS, TRAIN_SPANS)
AGAIN = SECOND.predict(TEXTS, ids=IDENTIFIERS)
same_run = [[mention.as_tuple() for mention in found] for found in BEFORE] == [
    [mention.as_tuple() for mention in found] for found in AGAIN
]
print("deux ajustements à graine fixée donnent les mêmes mentions :", same_run)
print(
    f"graine du projet : {CONFIG.seed} | époques : {NB_EPOCHS} | "
    f"messages d'entraînement : {len(TRAIN_DOCUMENTS)}"
)

variances = pd.DataFrame(
    {
        "run": ["premier ajustement", "second ajustement"],
        "mentions extraites": [
            sum(len(found) for found in BEFORE),
            sum(len(found) for found in AGAIN),
        ],
    }
)
display(variances)

**Ce qu'il faut retenir**

- Le déterminisme n'est pas un luxe : sans lui, aucune comparaison entre deux variantes n'a de sens — on ne saurait pas si l'écart vient du modèle ou du hasard.
- La graine est unique et vient de la configuration (`seed`), pas d'un appel caché à `random()`.
- La comparaison porte sur les mentions, pas sur les scores : deux modèles peuvent afficher la même F1 avec des erreurs différentes.

## 6. Comparer au run de référence

`make train` écrit ses artefacts dans `artifacts/`. S'il a tourné sur ce clone, on peut comparer le run de référence (corpus complet, budget d'époques du manifeste) au run du notebook — sans jamais confondre les deux.

In [ ]:
import json

REFERENCE_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "training_metrics.json"
key = f"val_{CONFIG.metrics.primary}"
notebook_score = float(OUTCOME.metrics[key])
if REFERENCE_PATH.exists():
    reference = json.loads(REFERENCE_PATH.read_text(encoding="utf-8"))
    reference_score = float(reference["metrics"].get(key, float("nan")))
    print(
        f"run de référence (make train) : {key} = {reference_score:.4f} "
        f"({reference['n_train_documents']} messages)"
    )
    print(
        f"run du notebook               : {key} = {notebook_score:.4f} "
        f"({OUTCOME.n_train_documents} messages, {NB_EPOCHS} époques)"
    )
    print(f"écart                         : {notebook_score - reference_score:+.4f}")
else:
    print("Aucun run de référence dans artifacts/ : lancer `make train` pour comparaison.")
    print(f"run du notebook : {key} = {notebook_score:.4f}")

**Ce qu'il faut retenir**

- Un notebook ne remplace pas un run de référence : corpus réduit et budget réduit servent à comprendre, pas à publier.
- L'écart entre les deux se lit comme un **coût du budget** : plus d'époques et plus de messages, plus de score — jusqu'à un plateau.
- La discipline reste la même : le split de test n'est ouvert qu'au notebook 06.

## Synthèse

| Étape | Objet | Artefact |
| --- | --- | --- |
| Génération | `DataGenerationPipeline` | `data/raw/*.parquet` (bac à sable) |
| Entraînement | `TrainPipeline` → `EntityTrainer` | `models/__MODEL_FILE__/` |
| Métriques | `EntityTrainingOutcome` | `metrics/training_metrics.json` |
| Traçabilité | `ModelCard` | `models/model_card.json` |
| Reproductibilité | `seed` de la configuration | `models/resolved_config.json` |

**Suite** : `06_error_analysis.ipynb` ouvre le test, une seule fois, et transforme les erreurs en recommandations.